In [26]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [27]:
from agents import set_default_openai_api, set_tracing_disabled
set_default_openai_api("chat_completions")
set_tracing_disabled(True)

In [28]:
os.environ["OPENAI_AGENTS_DISABLE_TRACING"] = "1"

In [29]:
os.environ["OPENAI_API_KEY"] = os.getenv("API_TOKEN")
os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"

In [30]:
from agents import Agent, Runner, function_tool

In [31]:
@function_tool
def get_weather(city: str) -> str:
    """
     Get the Current weather for a city

     Args:
     city: The name of the city to get the weather for

    """
    weather_data = {
        "london": "Cloudy, 15C",
        "tokyo": "Sunny, 22C",
        "new york": "Rainy, 18C"
    }
    return weather_data.get(city.lower(),f"Weather data no available for {city}")


In [32]:
weather_agent = Agent(
    name="WeatherBot",
    instructions="You help users check the weather, use the get_weather tool when asked about weather",
    model="gpt-4o-mini",
    tools=[get_weather]
)

In [33]:
runner = await Runner.run(weather_agent, "Whats the weather in sunderland")
print(runner.final_output)

I'm sorry, but I couldn't find the weather data for Sunderland. Would you like to check the weather for a different city?


Sequential tool calling

In [34]:
@function_tool
def get_user_id(email: str) -> str:
    """Look up user ID from email."""
    return "user_123"

@function_tool
def get_user_orders(user_id: str) -> str:
    """Get orders for a user ID."""
    return "Order #1: Laptop, Order #2: Mouse"

@function_tool
def get_order_status(order_id: str) -> str:
    """Get status of an order."""
    return "Shipped - arriving tomorrow"

agent = Agent(
    name="OrderAssistant",
    instructions="Help users check their order status. First find their user ID, then their orders, then status.",
    tools=[get_user_id, get_user_orders, get_order_status]
)

runner = await Runner.run(agent, "What's the status of my orders? My email is john@example.com")
print(runner.final_output)

Both orders are shipped and expected to arrive tomorrow:

- Laptop: Shipped
- Mouse: Shipped


Parallel tool calling

In [35]:
@function_tool
def get_weather(city: str) -> str:
    """Get weather for a city."""
    return f"{city}: Sunny, 25°C"

@function_tool
def get_news(topic: str) -> str:
    """Get latest news on a topic."""
    return f"Latest {topic} news: ..."

@function_tool
def get_stock(symbol: str) -> str:
    """Get stock price."""
    return f"{symbol}: $150.00"


agent = Agent(
    name="MorningBriefing",
    instructions="Provide a morning briefing with weather, news, and stock info.",
    tools=[get_weather, get_news, get_stock]
)

runner = await Runner.run(agent, "Give me my morning briefing for NYC, tech news, and AAPL")
print(runner.final_output)

## Morning briefing

- **NYC weather:** Sunny, **25°C**
- **Tech news:** Latest tech news: …
- **AAPL:** **$150.00**


Conditional tool calling

In [36]:
@function_tool
def search_web(query: str) -> str:
    """Search the web for current information."""
    return f"Web results for '{query}': ..."

@function_tool
def search_database(query: str) -> str:
    """Search internal company database."""
    return f"Database results for '{query}': ..."

@function_tool
def search_documents(query: str) -> str:
    """Search uploaded documents."""
    return f"Document results for '{query}': ..."

agent = Agent(
    name="SmartSearch",
    instructions="""You are a search assistant.
    
    Choose the right search based on the query:
    - For current events/general info → use search_web
    - For company/employee info → use search_database
    - For policy/procedure questions → use search_documents
    """,
    model="gpt-4o-mini",
    tools=[search_web, search_database, search_documents]
)

runner = await Runner.run(agent, "Find the latest company policy on remote work.")

print(runner.final_output)

I found the latest document regarding the company's remote work policy. Please provide specific details or aspects you would like to know about this policy.
